# Module 16: Social Media Newsfeed Recommendation TwoTower — Interactive Laboratory

Every cell below runs the module's **real** implementation from
`project_solution/newsfeed_recommendation_engine.py`. Nothing here prints a claim it has not verified.

What you will do:

1. Load the engine and inspect what it actually exports.
2. Run its primary workflow and check the assertions that define correctness.
3. **Commit to a prediction**, then run the cell that tests it.
4. Measure a property rather than asserting one.
5. Fix a deliberately broken cell in place.

> The code in cells 4, 6 and 8 is lifted from this module's own test suite, so it
> cannot drift from the implementation. If the API changes, those tests fail
> first and this notebook is regenerated from them.


## 1. Load the engine and introspect it

Rather than trusting a hardcoded list of class names, ask the module what it
actually contains.


In [ ]:
import inspect
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve() / 'project_solution'))
import newsfeed_recommendation_engine

classes = [n for n, o in inspect.getmembers(newsfeed_recommendation_engine, inspect.isclass)
           if o.__module__ == 'newsfeed_recommendation_engine']
functions = [n for n, o in inspect.getmembers(newsfeed_recommendation_engine, inspect.isfunction)
             if o.__module__ == 'newsfeed_recommendation_engine']

print('module   : newsfeed_recommendation_engine')
print(f'classes  : {classes}')
print(f'functions: {functions}')
print()
for name in classes:
    obj = getattr(newsfeed_recommendation_engine, name)
    try:
        sig = inspect.signature(obj.__init__)
        params = [p for p in sig.parameters if p != 'self']
    except (TypeError, ValueError):
        params = ['<builtin>']
    print(f'  {name}({", ".join(params)})')

## 2. Baseline: Standard user fanout on write

This is the module's own `test_standard_user_fanout_on_write` — real instantiation, real calls, real
assertions. If it runs clean, the property it encodes holds.


In [ ]:
from newsfeed_recommendation_engine import (
    HybridNewsfeedEngine,
    Post,
    UserProfile,
)

engine = HybridNewsfeedEngine(celebrity_threshold_followers=100)

alice = UserProfile("alice")
bob = UserProfile("bob")
engine.register_user(alice)
engine.register_user(bob)

# Bob follows Alice
engine.follow("bob", "alice")

post = Post(
    post_id="post-1",
    author_id="alice",
    content="Deploying microservices!",
    category="TECH",
    embedding=(1.0, 0.0, 0.0, 0.0),
)
engine.publish_post(post)

# Post must be pushed directly into Bob's timeline cache
assert len(engine.timeline_cache["bob"]) == 1
assert engine.timeline_cache["bob"][0].post_id == "post-1"

print('PASSED: test_standard_user_fanout_on_write')

## 3. 🔮 Prediction — commit before you run

Predict whether fan-out-on-write or fan-out-on-read is cheaper for a user with 50 million followers - and whether the answer flips for a user with 12.

Write your answer down. An uncommitted guess teaches nothing, because you will
retro-fit it to whatever the next cell prints.

The next cell runs `test_celebrity_fanout_on_read_merging`, which tests exactly this property.


In [ ]:
engine = HybridNewsfeedEngine(celebrity_threshold_followers=3)

elon = UserProfile("elon")
engine.register_user(elon)

# 3 users follow Elon -> triggers celebrity status
for i in range(1, 4):
    u = UserProfile(f"follower-{i}")
    engine.register_user(u)
    engine.follow(f"follower-{i}", "elon")

assert engine.users["elon"].is_celebrity is True

celeb_post = Post(
    post_id="tweet-mars",
    author_id="elon",
    content="Starship orbital test tomorrow!",
    category="SPACE",
    embedding=(0.0, 1.0, 0.0, 0.0),
)
engine.publish_post(celeb_post)

# Invariant: Celebrity post must NOT be duplicated across all followers' timeline caches!
assert len(engine.timeline_cache["follower-1"]) == 0
assert len(engine.celebrity_posts["elon"]) == 1

# When follower reads feed, celebrity post is dynamically merged!
feed = engine.get_chronological_feed("follower-1")
assert len(feed) == 1
assert feed[0].post_id == "tweet-mars"

print('PASSED: test_celebrity_fanout_on_read_merging')

## 4. Measure it: Two tower candidate retrieval and ranking

An assertion tells you a property holds. A measurement tells you *how much*.
This cell runs `test_two_tower_candidate_retrieval_and_ranking` and times it.


In [ ]:
import time

_t0 = time.perf_counter()

engine = HybridNewsfeedEngine()

# User with strong interest in TECH: (1.0, 0.0, 0.0, 0.0)
tech_user = UserProfile("techie", interest_vector=(1.0, 0.0, 0.0, 0.0))
engine.register_user(tech_user)

# Post A: Tech related (vector match = 1.0)
post_a = Post(
    post_id="p-tech",
    author_id="author-1",
    content="Kubernetes vs Nomad",
    category="TECH",
    embedding=(1.0, 0.0, 0.0, 0.0),
    likes=50,
)
# Post B: Cooking related (orthogonal vector match = 0.0)
post_b = Post(
    post_id="p-cooking",
    author_id="author-2",
    content="Best Pasta Carbonara Recipe",
    category="FOOD",
    embedding=(0.0, 1.0, 0.0, 0.0),
    likes=200,
)
engine.publish_post(post_a)
engine.publish_post(post_b)

# AI Personalized feed must rank the relevant Tech post first despite Cooking having more likes
ai_feed = engine.get_personalized_ai_feed("techie", limit=5)
assert len(ai_feed) == 2
assert ai_feed[0].post_id == "p-tech"
assert ai_feed[1].post_id == "p-cooking"

_elapsed = (time.perf_counter() - _t0) * 1000
print('PASSED: test_two_tower_candidate_retrieval_and_ranking')
print(f'wall clock: {_elapsed:.2f} ms')

## 5. 🛠️ Fix this cell — it is deliberately broken

The cell below asserts something **false** about the real object. Read the
failure, work out the true value from the module's actual behaviour, and correct
the expected number.

Do not delete the assertion. The point is to make it pass by knowing the answer.


In [ ]:
# DELIBERATELY BROKEN - fix the expected value below.
# Hint: print the real value first, then decide what the assertion should say.

exports = [n for n in dir(newsfeed_recommendation_engine) if not n.startswith('_')]
print(f'actual export count: {len(exports)}')
print(f'actual exports     : {exports}')

EXPECTED_EXPORT_COUNT = 999      # <-- wrong on purpose. Replace it.

assert len(exports) == EXPECTED_EXPORT_COUNT, (
    f'expected {EXPECTED_EXPORT_COUNT} exports, found {len(exports)}. '
    'Read the printed value above and correct the constant.'
)
print('Fixed - assertion now reflects reality.')

### 🎓 Key takeaways

1. Fan-out on write is cheap to read and expensive for celebrities.
2. Hybrid fan-out exists because neither pure strategy survives the tail.
3. Two-tower retrieval separates candidate generation from ranking for a reason.

---

**Continue with this module:**

- [README.md](README.md) — the mental model and failure modes
- [PROJECT_GUIDE.md](PROJECT_GUIDE.md) — build it yourself, in 3 tiers
- [starter/](starter/) — your stubs; run the tests from there to grade yourself
- [debug_lab/SYMPTOMS.md](debug_lab/SYMPTOMS.md) — diagnose planted bugs from the symptom
- [TROUBLESHOOTING_AND_EDGE_CASES.md](TROUBLESHOOTING_AND_EDGE_CASES.md) — real errors, real causes
- [SELF_ASSESSMENT_AND_CHALLENGES.md](SELF_ASSESSMENT_AND_CHALLENGES.md) — quiz and diagnostics
